# 信噪 · Qwen2.5-1.5B LoRA 微调（Kaggle 瘦壳）

这个 notebook 只是 `scripts/train_lora.py` 的 Kaggle 运行器。所有训练逻辑（超参、模型加载、SFTTrainer 配置）都在那个 .py 里——这样代码能 diff、能 import、能在任何 GPU 环境跑。

**如果你能直接传 .py 跑 Kaggle Script Kernel，更推荐那条路。** 这个 notebook 是为了 ① 看 cell 中间状态 ② 训练后立刻试推理。

## 准备
1. 把仓库代码（含 `scripts/train_lora.py`）作为 Kaggle Dataset 加进来——或者用 `!git clone` 现拉
2. 把 `data/sft_train.jsonl` + `data/sft_val.jsonl` 作为第二个 Dataset 加进来
3. Settings → Accelerator = GPU T4 x2 或 P100；Internet = On

**GPU 提示**：T4 / P100 不支持 bf16，训练脚本会自动切到 fp16。无需手动配置。

In [ ]:
# 1. 安装依赖
#
# Kaggle 预装的 transformers (5.8) / trl (1.4) / datasets (4.8) 版本太新，
# 超出 unsloth 的兼容范围 (transformers<=5.5, trl<=0.24, datasets<4.4)。
# 先卸掉这几个，再装 unsloth，让它自己拉兼容版本的依赖。
# peft / accelerate / bitsandbytes 会作为 unsloth 的依赖自动安装。
#
# 注意：装完可能要点 "Restart Session" 让 Python 重新加载，再跑后续 cell。
# 看到 "Successfully installed unsloth..." 即可。
!pip uninstall -y -q transformers trl datasets unsloth unsloth-zoo
!pip install -q unsloth bitsandbytes

In [ ]:
# 2. 找代码 / 数据路径，切到工作目录
import os, shutil
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/repo")

# 找含 train_lora.py 的代码目录
script_hits = list(INPUT_ROOT.rglob("scripts/train_lora.py"))
if script_hits:
    code_root = script_hits[0].parents[1]
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    shutil.copytree(code_root, WORK_DIR)
else:
    # 没传 dataset 就 clone 仓库
    WORK_DIR.parent.mkdir(exist_ok=True)
    !git clone https://github.com/alchosyn/npc-dialogue-ai-agent.git $WORK_DIR
os.chdir(WORK_DIR)
print(f"工作目录: {WORK_DIR}")

# 找训练数据
train_hits = list(INPUT_ROOT.rglob("sft_train.jsonl"))
assert train_hits, "找不到 sft_train.jsonl，请把数据上传为 Kaggle Dataset"
TRAIN_PATH = train_hits[0]
VAL_PATH = TRAIN_PATH.parent / "sft_val.jsonl"
OUTPUT_DIR = Path("/kaggle/working/qwen-1.5b-xinzao-lora")
print(f"train: {TRAIN_PATH}\nval:   {VAL_PATH}\nout:   {OUTPUT_DIR}")

# 检查 GPU 能力（提前提示是 bf16 还是 fp16）
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {name} (compute capability {cap[0]}.{cap[1]})")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# 3. 跑训练（所有逻辑都在 scripts/train_lora.py 里）
#    --precision auto：脚本自动按 GPU 能力选 bf16/fp16
#    --optim auto：bitsandbytes 可用就用 adamw_8bit，否则 adamw_torch
import subprocess, sys

cmd = [
    sys.executable, "scripts/train_lora.py",
    "--train-jsonl", str(TRAIN_PATH),
    "--val-jsonl", str(VAL_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--precision", "auto",
    "--optim", "auto",
    "--smoke-test",
]
print("+ " + " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 4. 看输出，确认 adapter 已经保存
!ls -lh /kaggle/working/qwen-1.5b-xinzao-lora
print("\n训完之后这个目录会作为 Kaggle Output 出现在右侧，可下载或转 Dataset 供 eval notebook 用")